In [ ]:
import pandas as pd
import os
from datetime import datetime

def sample_wildtypes(df, n=10, mutations_col='Mutations', id_col='Accession', 
                     species_col='Species', family_col='Opsin_Family',
                     exclusive_wt=False, save_files=False):
    """
    Samples 'n' wildtype sequences (where mutations are empty), 
    returns the sample and the remaining main dataframe.
    
    Args:
        df: The input dataframe.
        n: Number of wildtypes to sample.
        mutations_col: The column identifying mutations.
        id_col: The column containing the Accession (used to link WT to Mutants).
        species_col: Column for Species name.
        family_col: Column for Opsin Family.
        exclusive_wt: If True, only samples WT sequences that have no corresponding 
                      mutants based on Accession prefix OR Species/Family overlap.
        save_files: Boolean to toggle saving results to TSV.
    """
    # Preprocessing
    df['Full_Species'] = df['Species'].copy()
    df['Genus'] = [sp.split('_')[0] for sp in df['Species'].to_list()]
    
    species_list = []
    for sp in df['Species']:
        sp_slit = sp.split('_')[1:]
        sp_name = ''
        for part in sp_slit:
            sp_name+=part
        species_list.append(sp_name)
    df['Species'] = species_list
    
    # 1. Identify all Wildtypes (empty or NaN)
    wt_mask = df[mutations_col].fillna('').str.strip() == ''
    wt_df = df[wt_mask]
    wt_accessions = wt_df['Accession'].to_list()
    drop_wt_list = []
    # 2. Filter WT candidates
    if exclusive_wt:
        # --- A. Filter by Accession Prefix ---
        # Get all accessions that are mutants (have mutations)
        mutant_df = df[~wt_mask]
        mutant_accessions = mutant_df[id_col].to_list()
        
        for mut_acc in mutant_accessions:
            for wt in wt_accessions:
                if ((wt in mut_acc) or any(key_word in wt for key_word in ['Rh1','Anc','pigment','lws','anc'])) and (wt not in drop_wt_list):
                    drop_wt_list.append(wt)
        
        # --- B. Filter by Species + Family overlap ---
        # Get unique pairs of (Species, Family) that have mutants in the dataset
        mutant_groups = mutant_df[[species_col, family_col]].drop_duplicates()
        
        # Start with all wildtypes
        wt_candidates = df[wt_mask].copy()
        
        # Apply Accession filter
        wt_candidates = wt_candidates[~wt_candidates[id_col].isin(drop_wt_list)]
        
        # Apply Species/Family filter via a left merge
        # This removes any WT whose Species/Family combo appears in the mutant list
        merged = wt_candidates.merge(mutant_groups, on=[species_col, family_col], 
                                     how='left', indicator=True)
        wt_candidates = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
        
        # Reset index because merge can mess with it, but we need original indices for dropping
        # Actually, wt_candidates here is a new DF from merge, but we need to drop from original df.
        # Let's ensure we keep the original index.
        wt_candidates.index = merged.loc[merged['_merge'] == 'left_only'].index
        
    else:
        wt_candidates = df[wt_mask]
    
    # 3. Handle sampling count
    if len(wt_candidates) < n:
        print(f"Warning: Only {len(wt_candidates)} suitable wildtype sequences found. Sampling all available.")
        n = len(wt_candidates)

    if n == 0:
        print("No suitable wildtype sequences found based on criteria.")
        return pd.DataFrame(), df

    # 4. Randomly sample 'n' indices
    sampled_wt_df = wt_candidates.sample(n=n, random_state=42)
    sampled_wt_df = sampled_wt_df[['Seq_Id', 'Lambda_Max', 'Accession', 'Opsin_Family', 'Full_Species', 'Genus', 'Species', 'Phylum', 'Class', 'Protein', 'RefId']]
    # 5. Remove the sampled rows from the main dataframe
    # Use the 'Seq_Id' or internal index to ensure accurate removal
    main_df_updated = df.drop(sampled_wt_df.index)
    main_df_updated=main_df_updated[['Seq_Id', 'Lambda_Max', 'Accession', 'Mutations', 'Opsin_Family', 'Full_Species', 'Genus', 'Species', 'Phylum', 'Class', 'Protein', 'RefId']]
    
    # 6. Optional Step: Save to files
    if save_files:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        mode = "exclusive" if exclusive_wt else "standard"
        wt_filename = f"sampled_{mode}_wt_{timestamp}.tsv"
        main_filename = f"updated_main_{timestamp}.tsv"
        
        sampled_wt_df.to_csv(wt_filename, sep='\t', index=False)
        main_df_updated.to_csv(main_filename, sep='\t', index=False)
        
        print(f"Files saved successfully ({mode} mode):")
        print(f" - Wildtypes: {wt_filename}")
        print(f" - Updated Main: {main_filename}")
    
    return sampled_wt_df, main_df_updated

In [54]:
# --- Example Usage ---
file_path = './vpod_1.3_data_splits_2025-10-06_16-50-06/wds_meta.tsv'
try:
    df = pd.read_csv(file_path, sep='\t')

    # Sampling 50 wildtypes and saving the output
    n_to_sample = 50
    sampled_wt, main_df = sample_wildtypes(
        df, 
        n=n_to_sample, 
        exclusive_wt=True, 
        save_files=True
    )
    
    print(f"\nSuccessfully sampled {len(sampled_wt)} wildtype sequences.")
    print(f"Remaining sequences in main dataframe: {len(main_df)}")
    
except FileNotFoundError:
    print(f"Error: {file_path} not found. Please ensure the file is in the working directory.")

Files saved successfully (exclusive mode):
 - Wildtypes: sampled_exclusive_wt_20260311_130205.tsv
 - Updated Main: updated_main_20260311_130205.tsv

Successfully sampled 50 wildtype sequences.
Remaining sequences in main dataframe: 1161


In [55]:
main_df

,Seq_Id,Lambda_Max,Accession,Mutations,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId
0,Bovine,500.0,NM_001014890,NaN,Rh1,Bos_tarus,Bos,tarus,Chordata,Mammalia,MNGTEGPNFYVPFSNKTGVVRSPFEAPQYYLAEPWQFSMLAAYMFL...,NaN
1,S1,502.0,U57536.1,NaN,Rh1,Neoniphon_sammara,Neoniphon,sammara,Chordata,Actinopteri,MNGTEGPYFYVPMVNTTGVVRSPYEYPQYYLVNPAAFAVLGAYMFF...,160.0
2,S2,502.0,U57540.1,NaN,Rh1,Neoniphon_argenteus,Neoniphon,argenteus,Chordata,Actinopteri,TEGPYFYVPMVNTTGIVRSPYEYPQYYLVNPAAYAVLGAYMFFLII...,160.0
3,S3,481.0,U57541.1,NaN,Rh1,Neoniphon_aurolineatus,Neoniphon,aurolineatus,Chordata,Actinopteri,TEGPDFYIPMVNTSGLVRSPYEYPQYYLVNPAAFAFLGAYMFFLII...,160.0
4,S4,494.0,U57543.1,NaN,Rh1,Sargocentron_punctatissimum,Sargocentron,punctatissimum,Chordata,Actinopteri,TEGPFFYIPMVNTSGVVRSPYEYPQYYLVNPAAYAILGAYMFFLII...,160.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1206,S1206,501.0,GnathostomataAncRh1,NaN,Rh1,GnathostomataAnc_sp.,GnathostomataAnc,sp.,Chordata,Ancestor,MNGTEGENFYVPMSNKTGVVRSPFEYPQYYLAEPWKYSALAAYMFF...,587.0
1207,S1207,501.0,OsteichthyesAncRh1,NaN,Rh1,OsteichthyesAnc_sp.,OsteichthyesAnc,sp.,Chordata,Ancestor,MNGTEGPNFYVPMSNKTGVVRSPFEYPQYYLAEPWKYSALAAYMFF...,587.0
1208,S1208,501.0,RhipidistiaAncRh1,NaN,Rh1,RhipidistiaAnc_sp.,RhipidistiaAnc,sp.,Chordata,Ancestor,MNGTEGPNFYVPMSNKTGVVRSPFEYPQYYLAEPWKYSALAAYMFF...,587.0
1209,S1209,499.0,TetrapodaAncRh1,NaN,Rh1,TetrapodaAnc_sp.,TetrapodaAnc,sp.,Chordata,Ancestor,MNGTEGPNFYVPMSNKTGVVRSPFEYPQYYLAEPWKYSALAAYMFL...,587.0


In [56]:
sampled_wt

,Seq_Id,Lambda_Max,Accession,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId
62,S234,358.0,U63972.1,SWS1,Rattus_norvegicus,Rattus,norvegicus,Chordata,Mammalia,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,132.0
165,S684,507.0,OP722950.1,IV-LWS,Chrysochroa_mniszechii,Chrysochroa,mniszechii,Arthropoda,Insecta,MSALGEPNFAAWSAQRVMSGAFGGNYTVVDKVPPEMLYLVDHHWYQ...,418.0
95,S310,516.0,AF132042.1,MWS,Cavia_porcellus,Cavia,porcellus,Chordata,Mammalia,MAQRWGPHALSGVQAQDAYEDSTQASLFTYTNSNNTRGPFEGPNYH...,155.0
172,S729,532.0,OK930069.1,IV-LWS,Automeris_io,Automeris,io,Arthropoda,Insecta,MTISLDPGPGLAALQAWGGQVAAYGAANQTVVDKVPPDMLHMVDAH...,416.0
134,S436,486.0,LC260050,Rh2,Oryzias_luzonensis,Oryzias,luzonensis,Chordata,Actinopteri,MGWDGGEQNGTEGKNFYIPMSNRTGVVRSPYEYPQYYMVDPIMFKI...,367.0
180,S755,539.0,AB081277.2,LWS,Aotus_azaraiboliviensis,Aotus,azaraiboliviensis,Chordata,Lepidosauria,MAQQWSLQRLAGRHPQDNHEDSTQSSIFTYTNSNSTRGPFEGPNYH...,384.0
18,S19,498.0,AB084930.1,Rh1,Cyprichromis_leptosoma,Cyprichromis,leptosoma,Chordata,Actinopteri,MANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFFLMLVGFPINFLT...,163.0
22,S23,488.0,AB185221.1,Rh1,Greenwoodochromis_bellcrossi,Greenwoodochromis,bellcrossi,Chordata,Actinopteri,MTNTTGVIRSPYEYPQHYLVSPAAYAALGAYMFFLIIVGFPINFLT...,163.0
19,S20,502.0,AB084931.1,Rh1,Dimidiochromis_compressiceps,Dimidiochromis,compressiceps,Chordata,Actinopteri,MVNTTGIVRSPYEYPQHYLVSPAAYAALGAYMFFLILVGFPINFLT...,163.0
9,S10,499.0,U57539.1,Rh1,Myripristis_violacea,Myripristis,violacea,Chordata,Actinopteri,TEGPYFYIPMSNATGIVRSPYEYPQYYLVYPAAYAVLGAYMFFLII...,160.0
